# Acessos SciELO
Este arquivo tem como objetivo **facilitar a obtenção de dados de acesso aos periódicos SciELO**. A SciELO SUSHI API é consultada para cada ano, volume e números do periódico definidos pelo usuário. Ao final, **um arquivo Excel** (formato xlsx) é disponibilizado para ser baixado. Segue o passo a passo para definir os dados e obter as informaçoes.

1. Clique na opção `Executar tudo`
2. Defina na seção 2, os dados do periódico e do período de acessos desejado
3. Clique no botão `Obter dados` e aguarde o processo de coleta
4. Clique no botão `Baixar dados`


## 1. Definição de funções para obter dados

Esta seção define os mecanismos para obter os dados de acesso. O usuário pode consultar os detalhes ao clicar no botão `mostrar código`.

In [1]:
## Carrega bibliotecas essenciais para o funcionamento do código
import requests
import ipywidgets as widgets
import pandas as pd

from bs4 import BeautifulSoup
from google.colab import files
from IPython.display import display


## Define variáveis globais para serem usadas nas consultas
colecao_acronimo_g = 'scl'
revista_titulo_g = ''
revista_acronimo_g = ''
revista_ano_g = ''
revista_volume_g = ''
revista_numeros_g = ''
acessos_data_inicio_g = ''
acessos_data_fim_g = ''


## Define interface para usuário digitar dados
colecao_acronimo_widget = widgets.Text(
    value='scl',
    description='Acrônimo da Coleção:',
    disabled=False
)

revista_titulo_widget = widgets.Text(
    value='Revista CEFAC',
    description='Título da Revista:',
    disabled=False
)

revista_acronimo_widget = widgets.Text(
    value='rcefac',
    description='Acrônimo da Revista:',
    disabled=False
)

revista_ano_widget = widgets.IntText(
    value=2024,
    description='Ano da Revista:',
    disabled=False
)

revista_volume_widget = widgets.IntText(
    value=26,
    description='Volume da Revista:',
    disabled=False
)

revista_numeros_string_widget = widgets.Text(
    value='1, 2, 3, 4, 5, 6',
    description='Números da Revista (separados por vírgula):',
    disabled=False
)

acessos_data_inicio_widget = widgets.Text(
    value='2024-01-01',
    description='Data de Início dos Acessos (YYYY-MM-DD):',
    disabled=False
)

acessos_data_fim_widget = widgets.Text(
    value='2024-12-31',
    description='Data de Fim dos Acessos (YYYY-MM-DD):',
    disabled=False
)

def definir_variaveis(colecao_acronimo, revista_titulo, revista_acronimo, revista_ano, revista_volume, revista_numeros_string, acessos_data_inicio, acessos_data_fim):
    global colecao_acronimo_g, revista_titulo_g, revista_acronimo_g, revista_ano_g, revista_volume_g, revista_numeros_g, acessos_data_inicio_g, acessos_data_fim_g

    colecao_acronimo_g = colecao_acronimo
    revista_titulo_g = revista_titulo
    revista_acronimo_g = revista_acronimo
    revista_ano_g = revista_ano
    revista_volume_g = revista_volume

    try:
        revista_numeros_g = [int(n.strip()) for n in revista_numeros_string.split(',')]
    except ValueError:
        print("Erro: Certifique-se de que os números estejam separados por vírgula e sejam números inteiros.")

    acessos_data_inicio_g = acessos_data_inicio
    acessos_data_fim_g = acessos_data_fim

## 2. Defição dos dados do periódico e o intervalo de datas de acesso desejadas

**[Ação do usuário]** Preencha os campos seguintes.

In [2]:
# @title
widgets.interactive(definir_variaveis,
                    colecao_acronimo=colecao_acronimo_widget,
                    revista_titulo=revista_titulo_widget,
                    revista_acronimo=revista_acronimo_widget,
                    revista_ano=revista_ano_widget,
                    revista_volume=revista_volume_widget,
                    revista_numeros_string=revista_numeros_string_widget,
                    acessos_data_inicio=acessos_data_inicio_widget,
                    acessos_data_fim=acessos_data_fim_widget)

interactive(children=(Text(value='scl', description='Acrônimo da Coleção:'), Text(value='Revista CEFAC', descr…

## 3. Obtenção de dados

In [3]:
# @title
def get_and_process(b):
    ## Obtém códigos de artigos
    acessos_url = 'https://usage.apis.scielo.org/reports/ir_a1'
    acessos_params = {
        'collection': colecao_acronimo_g,
        'begin_date': acessos_data_inicio_g,
        'end_date': acessos_data_fim_g,
        'fmt': 'json',
        'api': 'v2'
    }

    revista_url = 'https://www.scielo.br/j/{0}/i/{1}.v{2}n{3}'

    numero_pids = {}

    for revista_numero in revista_numeros_g:
        numero_pids[revista_numero] = set()

        response = requests.get(revista_url.format(revista_acronimo_g, revista_ano_g, revista_volume_g, revista_numero))
        if response.status_code == 200:
            soup = BeautifulSoup(response.text, 'html.parser')

            links = soup.find_all('a', href=True)
            for link in links:
                href = link['href']
                if f'/j/{revista_acronimo_g}/a/' in href:
                    pid_v3 = href.split('/')[4]
                    numero_pids[revista_numero].add(pid_v3)

    print('---')
    print('Quantitativo de artigos a serem considerados na consulta')
    print('Número\tArtigos')
    for n in numero_pids:
        print(f'{n}\t{len(numero_pids[n])}')


    ## Obtém acessos da SciELO SUSHI API
    dados = {}
    total = sum([len(x) for x in numero_pids.values()])
    cum = 0

    print('---')
    for n, p in numero_pids.items():
        dados[n] = []

        for x, pid in enumerate(sorted(p)):
            cum += 1
            print(f'Obtendo acessos de artigo {cum} de {total}')
            res = requests.get(acessos_url, params={**acessos_params, 'pid': pid})
            dados[n].append(res.json())


    ## Processa dados
    rows = []

    for numero, artigos in dados.items():
        for a in artigos:
            try:
                acessos = a['Report_Items']
            except KeyError:
                continue

            for p in acessos:
                pid = ','.join(p['Item_ID'])
                for per in p['Performance']:
                    metric = per['Instance']['Metric_Type']
                    if metric != 'Total_Item_Requests':
                        continue

                    month = per['Period']['Begin_Date'][:7]  # Ano-mês (YYYY-MM)
                    count = per['Instance']['Count']

                    rows.append({
                        'ano': revista_ano_g,
                        'numero': numero,
                        'artigo': pid,
                        'mes': month,
                        'acessos': count
                    })

    df_acessos = pd.DataFrame(rows)


    ## Transforma dados para formato que facilita análise
    df_pivot = df_acessos.pivot_table(
        index=['ano', 'numero', 'artigo'],
        columns='mes',
        values='acessos',
        aggfunc='sum',
        fill_value=0
    )

    for col in df_pivot.columns:
        if col.startswith('2024-'):
            df_pivot[col] = df_pivot[col].astype(int)

    df_pivot['total'] = df_pivot.sum(axis=1)


    ## Grava dados em arquivo
    df_pivot.to_excel('acessos.xlsx')

    print('---')
    print('Concluído. Clique no botão `Baixar dados.xlsx`.')


## Definição de interface para obter e processar dados
botao_obter_e_processar = widgets.Button(description="Obter dados")
botao_obter_e_processar.on_click(get_and_process)


## Definição de interface para baixar resultados
botao_baixar = widgets.Button(description="Baixar acessos.xlsx")

def baixar_arquivo(b):
    try:
        files.download('acessos.xlsx')
        print("Arquivo baixado com sucesso!")
    except Exception as e:
        print(f"Erro ao baixar o arquivo: {e}")

botao_baixar.on_click(baixar_arquivo)

display(botao_obter_e_processar)

Button(description='Obter dados', style=ButtonStyle())

---
Quantitativo de artigos a serem considerados na consulta
Número	Artigos
1	10
2	10
3	10
4	10
5	10
6	11
---
Obtendo acessos de artigo 1 de 61
Obtendo acessos de artigo 2 de 61
Obtendo acessos de artigo 3 de 61
Obtendo acessos de artigo 4 de 61
Obtendo acessos de artigo 5 de 61
Obtendo acessos de artigo 6 de 61
Obtendo acessos de artigo 7 de 61
Obtendo acessos de artigo 8 de 61
Obtendo acessos de artigo 9 de 61
Obtendo acessos de artigo 10 de 61
Obtendo acessos de artigo 11 de 61
Obtendo acessos de artigo 12 de 61
Obtendo acessos de artigo 13 de 61
Obtendo acessos de artigo 14 de 61
Obtendo acessos de artigo 15 de 61
Obtendo acessos de artigo 16 de 61
Obtendo acessos de artigo 17 de 61
Obtendo acessos de artigo 18 de 61
Obtendo acessos de artigo 19 de 61
Obtendo acessos de artigo 20 de 61
Obtendo acessos de artigo 21 de 61
Obtendo acessos de artigo 22 de 61
Obtendo acessos de artigo 23 de 61
Obtendo acessos de artigo 24 de 61
Obtendo acessos de artigo 25 de 61
Obtendo acessos de artig

### Após a coleta estar concluída, clique no botão `Baixar acessos.xlsx`

In [4]:
# @title
display(botao_baixar)

Button(description='Baixar acessos.xlsx', style=ButtonStyle())

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Arquivo baixado com sucesso!
